scripts to process data in ray's metrics

In [1]:
import os
import glob
import pprint

EXP_PATH  = "../exp7/"

In [2]:
import Kkit

In [3]:
import pandas as pd
import numpy as np

In [4]:
metrics_path = glob.glob(os.path.join(EXP_PATH, "evaluation_metrics.pkl_*"))
Kkit.print_list(metrics_path, 4)

../exp7/evaluation_metrics.pkl_26 ,  ../exp7/evaluation_metrics.pkl_6 , ../exp7/evaluation_metrics.pkl_34 , ../exp7/evaluation_metrics.pkl_38
../exp7/evaluation_metrics.pkl_15 , ../exp7/evaluation_metrics.pkl_18 ,  ../exp7/evaluation_metrics.pkl_5 , ../exp7/evaluation_metrics.pkl_33
../exp7/evaluation_metrics.pkl_11 ,  ../exp7/evaluation_metrics.pkl_9 , ../exp7/evaluation_metrics.pkl_30 , ../exp7/evaluation_metrics.pkl_14
../exp7/evaluation_metrics.pkl_23 ,  ../exp7/evaluation_metrics.pkl_4 , ../exp7/evaluation_metrics.pkl_10 , ../exp7/evaluation_metrics.pkl_29
../exp7/evaluation_metrics.pkl_25 , ../exp7/evaluation_metrics.pkl_35 ,  ../exp7/evaluation_metrics.pkl_2 , ../exp7/evaluation_metrics.pkl_12
 ../exp7/evaluation_metrics.pkl_1 , ../exp7/evaluation_metrics.pkl_19 , ../exp7/evaluation_metrics.pkl_37 , ../exp7/evaluation_metrics.pkl_24
../exp7/evaluation_metrics.pkl_20 ,  ../exp7/evaluation_metrics.pkl_7 , ../exp7/evaluation_metrics.pkl_16 , ../exp7/evaluation_metrics.pkl_36
../exp

In [5]:
path_dict = {p: int(p.split("_")[-1]) for p in metrics_path}
path_dict = Kkit.sort_dic_by_value(path_dict)
metrics_path = list(path_dict.keys())
Kkit.print_list(metrics_path, 4)

 ../exp7/evaluation_metrics.pkl_1 ,  ../exp7/evaluation_metrics.pkl_2 ,  ../exp7/evaluation_metrics.pkl_3 ,  ../exp7/evaluation_metrics.pkl_4
 ../exp7/evaluation_metrics.pkl_5 ,  ../exp7/evaluation_metrics.pkl_6 ,  ../exp7/evaluation_metrics.pkl_7 ,  ../exp7/evaluation_metrics.pkl_8
 ../exp7/evaluation_metrics.pkl_9 , ../exp7/evaluation_metrics.pkl_10 , ../exp7/evaluation_metrics.pkl_11 , ../exp7/evaluation_metrics.pkl_12
../exp7/evaluation_metrics.pkl_13 , ../exp7/evaluation_metrics.pkl_14 , ../exp7/evaluation_metrics.pkl_15 , ../exp7/evaluation_metrics.pkl_16
../exp7/evaluation_metrics.pkl_17 , ../exp7/evaluation_metrics.pkl_18 , ../exp7/evaluation_metrics.pkl_19 , ../exp7/evaluation_metrics.pkl_20
../exp7/evaluation_metrics.pkl_21 , ../exp7/evaluation_metrics.pkl_22 , ../exp7/evaluation_metrics.pkl_23 , ../exp7/evaluation_metrics.pkl_24
../exp7/evaluation_metrics.pkl_25 , ../exp7/evaluation_metrics.pkl_26 , ../exp7/evaluation_metrics.pkl_27 , ../exp7/evaluation_metrics.pkl_28
../exp

In [6]:
import re

class RegexDict(dict):
    def __init__(self, *args, **kwargs):
        super(RegexDict, self).__init__(*args, **kwargs)
        for key, value in self.items():
            if isinstance(value, dict):
                self[key] = RegexDict(value)
        self.__abbrdict_keys = list(super(RegexDict, self).keys())
        for i in self.__abbrdict_keys:
            if isinstance(i, str)==False:
                raise TypeError("all keys of AbbrDict must be str")

    def __getitem__(self, key):
        keys = list(super(RegexDict, self).keys())
        filtered = [k for k in keys if re.match(key, k)]
        return {k: super(RegexDict, self).__getitem__(k) for k in filtered}

In [7]:
metrics = []
for path in metrics_path:
    metrics.append(Kkit.load(path))
pprint.pprint(metrics[0])

{'env_runners': {'connector_metrics': {'ObsPreprocessorConnector_ms': 0.009205341339111328,
                                       'StateBufferConnector_ms': 0.00494082768758138,
                                       'ViewRequirementAgentConnector_ms': 0.09917497634887695},
                 'custom_metrics': {'reconf_mean_agent_0': 236.3,
                                    'reconf_mean_agent_1': 273.2,
                                    'reconf_mean_agent_2': 257.1,
                                    'reconf_sem_agent_0': 6.3176120664553475,
                                    'reconf_sem_agent_1': 2.5245461638348647,
                                    'reconf_sem_agent_2': 2.3965252624024918,
                                    'reconf_times_agent_0': [239,
                                                             233,
                                                             255,
                                                             259,
                            

In [23]:
np.mean([np.array(v) for v in RegexDict(metrics[0]["env_runners"]["custom_metrics"])["^reconf_times_agent_"].values()], axis=0)

array([257.        , 255.33333333, 263.33333333, 266.        ,
       255.        , 249.33333333, 236.33333333, 243.        ,
       268.33333333, 261.66666667])

In [27]:
reconf_times = []
sla_vio_times = []
reward = []
for metric in metrics:
    reconf = np.mean([np.array(v) for v in RegexDict(metric["env_runners"]["custom_metrics"])["^reconf_times_agent_"].values()], axis=0)
    sla_vio = np.mean([np.array(v) for v in RegexDict(metric["env_runners"]["custom_metrics"])["^sla_vio_times_agent_"].values()], axis=0)
    rew = metric["env_runners"]["hist_stats"]["episode_reward"]

    reconf_times.append(reconf)
    sla_vio_times.append(sla_vio)
    reward.append(rew)

In [29]:
reconf_times[30], sla_vio_times[30], reward[30]

(array([16.33333333, 12.66666667, 20.        , 20.66666667, 21.        ,
        13.33333333, 17.        , 16.        , 31.        , 17.33333333]),
 array([ 4.66666667,  5.33333333, 13.33333333, 10.        , 13.66666667,
         3.        ,  6.66666667,  5.33333333, 14.33333333,  6.33333333]),
 [7963.476472991639,
  8693.12288312182,
  8671.263928170612,
  8189.640930190789,
  8921.354752861964,
  9080.13039737141,
  8433.374845342321,
  8729.470159371514,
  8638.970846240165,
  7068.435092255907])

In [31]:
sla_df = pd.DataFrame(sla_vio_times)
sla_df

,0,1,2,3,4,5,6,7,8,9
0,139.333333,116.666667,150.666667,107.000000,134.333333,132.666667,121.333333,114.666667,111.333333,120.333333
1,47.666667,26.333333,19.333333,19.666667,18.333333,34.333333,10.333333,23.000000,36.000000,32.666667
2,16.000000,12.333333,34.333333,20.666667,47.333333,48.666667,27.666667,17.000000,23.333333,17.666667
3,7.000000,6.000000,10.666667,13.000000,6.666667,12.000000,6.000000,11.666667,8.000000,9.000000
4,10.000000,20.666667,19.333333,20.000000,21.000000,24.333333,18.666667,18.333333,22.000000,10.333333
5,9.333333,17.000000,6.000000,20.000000,16.333333,24.333333,14.000000,5.666667,14.333333,5.000000
6,10.333333,16.000000,25.000000,22.666667,36.000000,19.000000,18.666667,16.666667,51.333333,14.000000
7,9.333333,29.666667,20.333333,24.333333,30.666667,46.000000,20.000000,28.000000,16.666667,24.333333
8,8.666667,39.000000,17.333333,20.333333,28.333333,13.666667,46.333333,9.666667,26.000000,22.333333
9,7.666667,14.333333,20.666667,27.666667,14.666667,11.666667,21.000000,12.666667,13.666667,21.000000


In [32]:
reconf_df = pd.DataFrame(reconf_times)
reconf_df

,0,1,2,3,4,5,6,7,8,9
0,257.000000,255.333333,263.333333,266.000000,255.000000,249.333333,236.333333,243.000000,268.333333,261.666667
1,74.000000,79.666667,89.000000,80.000000,77.333333,97.000000,61.666667,79.666667,81.000000,83.333333
2,70.000000,68.333333,117.000000,75.666667,129.333333,131.666667,93.000000,91.333333,72.333333,82.666667
3,61.666667,67.333333,84.333333,44.000000,37.666667,53.000000,57.666667,58.666667,67.333333,72.000000
4,46.000000,92.333333,117.666667,68.666667,88.000000,104.666667,55.000000,82.333333,86.000000,66.333333
5,66.666667,54.000000,72.000000,70.000000,77.333333,110.666667,77.000000,69.333333,83.333333,66.666667
6,38.000000,51.000000,81.333333,60.333333,88.666667,92.000000,91.666667,86.000000,128.333333,99.666667
7,32.000000,65.333333,65.000000,46.000000,50.666667,73.000000,57.333333,34.666667,42.333333,61.666667
8,38.333333,95.666667,48.000000,58.333333,72.333333,72.666667,74.000000,28.333333,101.000000,62.000000
9,34.000000,34.666667,69.666667,73.333333,41.666667,48.000000,55.000000,39.000000,45.333333,70.000000


In [33]:
rew_df = pd.DataFrame(reward)
rew_df

,0,1,2,3,4,5,6,7,8,9
0,-5550.595939,-5419.555721,-6415.967860,-5505.328053,-5743.335117,-6154.082547,-5150.817323,-5387.160257,-6262.685329,-5770.310860
1,2410.381945,2505.809401,2243.878614,2368.821147,1615.141662,675.914632,1624.590454,1845.865938,1606.188507,2928.997499
2,5055.075430,4945.726153,4320.723337,4592.674996,2248.253851,3639.502602,3543.995145,4790.016168,4554.241288,4450.849074
3,4967.696364,6091.044431,5907.956942,5524.147318,6821.322766,7614.955095,5372.332189,6588.124553,6813.634632,6084.017248
4,5986.677875,6428.317711,5944.434245,6114.146992,5793.457551,6788.996099,5711.200965,6083.163494,6436.631799,6263.046403
5,6778.617413,6156.034064,7127.471971,6513.280240,6408.684224,6283.426587,5771.648935,7914.274413,7272.119792,6817.682011
6,7302.648976,6895.512259,4894.953997,6756.782828,6221.325936,7952.111852,7030.969757,8069.570534,5917.080623,7902.708145
7,7725.174221,6784.091825,6286.455230,6548.240921,6042.023039,6909.991508,7787.909954,8147.860811,8651.352266,7833.070831
8,7213.461231,7522.967913,6857.372691,7079.606551,5894.856049,7259.034106,6847.806735,8466.782638,7422.432129,8282.337103
9,7756.277672,8273.894003,6123.171275,7747.124446,7639.189998,6338.156985,8148.911872,8489.939972,7695.996321,8438.684360


In [34]:
# store all data
sla_df.to_csv(os.path.join(EXP_PATH, "sla_vio_times.csv"), index=False)
reconf_df.to_csv(os.path.join(EXP_PATH, "reconf_times.csv"), index=False)
rew_df.to_csv(os.path.join(EXP_PATH, "reward.csv"), index=False)